In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [60]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [#'pachterlab_sleuth' 
#            'geostat-framework_gstools' 
#            'juliaearth_geostats.jl' 
 #          'jaredhuling_oem'
 #           'avehtari_ros-examples'
#           'scikit-hep_uproot4'
#            'juliaphysics_measurements.jl'
 #          'asfhyp3_hyp3'
 #           'daehwankimlab_hisat2' 
           'hjkgrp_molsimplify'
           ]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,000fd53e7967f3246964daef111822a332925c5c,hjkgrp_molsimplify


In [61]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

100%|████████████████████████████████████████████████████████████████████████████████| 404/404 [06:52<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000fd53e7967f3246964daef111822a332925c5c,9366e45f6a1fd571c493a79d5b021821afaa5c12,"[7af10554d8551533be864bda7ce12cec999be923, 1c3...",Ralf Meyer <meyer.ralf@yahoo.com>,1682607679,-0400,GitHub <noreply@github.com>,1682607679,-0400,Merge branch 'master' into code_quality,000fd53e7967f3246964daef111822a332925c5c,hjkgrp_molsimplify
1,0045d14614d3efbe30ffd5ca9c7a0f0668c89b29,870d77d4b0760ea2f68e35ce9cf4f03d5a89b928,[d1d682ecd94478e899546345ddbc510e2ef15be7],chenruduan <duanchenru@gmail.com>,1590499886,-0400,chenruduan <duanchenru@gmail.com>,1590499886,-0400,Update molscontrol and its examples/tests.\n,0045d14614d3efbe30ffd5ca9c7a0f0668c89b29,hjkgrp_molsimplify


In [62]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '000fd53e7967f3246964daef111822a332925c5c', 'tree': '9366e45f6a1fd571c493a79d5b021821afaa5c12', 'parent': ['7af10554d8551533be864bda7ce12cec999be923', '1c39c62d6cef282c77568ad46ec7ec9b3c04f839'], 'author': 'Ralf Meyer <meyer.ralf@yahoo.com>', 'author_time': 1682607679, 'author_tz': '-0400', 'committer': 'GitHub <noreply@github.com>', 'committer_time': 1682607679, 'committer_tz': '-0400', 'message': "Merge branch 'master' into code_quality"}


In [63]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [64]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,hjkgrp_molsimplify,000fd53e7967f3246964daef111822a332925c5c,Ralf Meyer <meyer.ralf@yahoo.com>,1682607679,Merge branch 'master' into code_quality


In [65]:
#mode a means append, so you have all your projects in the same file
yournetid='afink12'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

# Github Info
## repo name: [number of stars, number of forks, last commit date]

<pre>
pachterlab_sleuth:            [317 stars, 98 forks, 28 May 2025]  
geostat-framework_gstools:    [652 stars, 83 forks, 03 July 2026]  
juliaearth_geostats.jl:       [590 stars, 66 forks, 15 Sept 2026]  
jaredhuling_oem:              [27 stars, 6 forks, 27 July 2024]  
avehtari_ros-examples:        [357 stars, 255 forks, 18 May 2025]  
scikit-hep_uproot4:           [270 stars, 102 forks, 22 Sept 2026]  
juliaphysics_measurements.jl: [536 stars, 43 forks, 22 Sept 2026]  
asfhyp3_hyp3:                 [53 stars, 13 forks, 23 Sept 2026]  
daehwankimlab_hisat2:         [542 stars, 131 forks, 31 July 2026]  
hjkgrp_molsimplify:           [227 stars, 60 forks, 18 Aug 2026]  

</pre>

# WoC Api Info
## repo name: [number of commits, number of authors, max time, min time]

<pre>
pachterlab_sleuth:            [861 commits, 50 authors, 2025-11-05 20:06:08 UTC, 2015-02-13 18:44:25 UTC]      
geostat-framework_gstools:    [2547 commits, 27 authors, 2025-11-03 16:15:01 UTC, 2018-01-15 11:04:09 UTC]      
juliaearth_geostats.jl:       [3243 commits, 34 authors, 2025-10-28 12:37:54 UTC, 2015-04-12 18:15:06 UTC]     
jaredhuling_oem:              [534 commits, 8 authors, 2024-07-27 16:43:51 UTC, 2016-03-28 14:37:23 UTC]     
avehtari_ros-examples:        [633 commits, 10 authors, 2025-05-21 18:22:17 UTC, 2018-02-05 19:52:41 UTC]     
scikit-hep_uproot4:           [5359 commits, 125 authors, 2025-11-05 20:06:08 UTC, 2020-05-08 20:30:10 UTC]  
juliaphysics_measurements.jl: [976 commits, 51 authors, 2026-04-20 13:52:43 UTC, 2016-05-16 19:06:38 UTC]    
asfhyp3_hyp3:                 [9010 commits, 37 authors, 2025-11-03 23:33:57 UTC, 2020-04-29 17:33:48 UTC]      
daehwankimlab_hisat2:         [1954 commits, 65 authors, 2025-10-14 15:44:56 UTC, 2015-03-30 18:20:21 UTC]      
hjkgrp_molsimplify:           [4039 commits, 121 authors, 2025-11-03 20:58:35 UTC, 2016-03-17 13:43:08 UTC]    

</pre>
